# Notebook 1b — Results Comparison (Tables Only)

**Purpose:** Paper-ready **tabular** presentation of the same final Best Result scores shown in `results_comparison.ipynb`, with rounding appropriate for a research paper.

**Input:** One `stdout.log` file per experiment (produced by `train.py` / `test.py` with the `_Tee` redirect).

**Definitions (as reported by the DCASE evaluator):**
- **F-measure (Harmonic)** — the official DCASE ranking metric. TP / FP / FN are pooled *within* each of the six reference subsets to give six per-subset F-measures; those six are then combined with a **harmonic** mean.
- **F-measure (Arithmetic)** — the **arithmetic** mean of those same six per-subset F-measures. Both columns average over *subsets*, not over classes; they differ only in the choice of mean, and the arithmetic value is always $\ge$ the harmonic one (AM–HM inequality).

**Outputs (all tabular):**
1. Summary table — harmonic & arithmetic subset-mean F-measure, precision, recall (2-decimal, 0–100 scale)
2. F-measure ranking table (harmonic subset mean)
3. F-measure ranking table (arithmetic subset mean)
4. Precision / Recall / F-measure per experiment (harmonic subset mean)
5. No-postproc vs. best-postproc F-measure + gain
6. Harmonic vs. arithmetic subset-mean F-measure with gap
7. LaTeX export of the main summary table

## 0 — Configuration

Edit `EXPERIMENT_LOGS` to point to your log files. Leave a value as `None` to skip an experiment.

In [1]:
import sys
from pathlib import Path

# ── Add notebooks/ to path so parse_logs can be imported ──────────────
sys.path.insert(0, str(Path(".").resolve()))
import parse_logs as pl

# ── Experiment log paths ────────────────────────────────────
# Fill in the actual paths to each run's stdout.log.
# Leave the value as None to skip an experiment.
EXPERIMENT_LOGS = {
    "Baseline":            "/media/helios/Masters/Repository/dcase-few-shot-bioacoustic/baselines/dcase2024_task5/logs/experiments/runs/baseline_seed1234/2026-09-17_09-08-46/stdout.log",  # e.g. "../logs/experiments/runs/baseline/2026-.../stdout.log"
    "CIFE":                "/media/helios/Masters/Repository/dcase-few-shot-bioacoustic/baselines/dcase2024_task5/logs/experiments/runs/cife_seed1234/2026-09-17_22-11-22/stdout.log",
    "Slot Uniform":        "/media/helios/Masters/Repository/dcase-few-shot-bioacoustic/baselines/dcase2024_task5/logs/experiments/runs/slot_uniform_seed1234/2026-09-18_12-05-18/stdout.log",
    "Phase 1 (Random)":    "/media/helios/Masters/Repository/dcase-few-shot-bioacoustic/baselines/dcase2024_task5/logs/experiments/runs/phase1_random_seed1234/2026-09-18_05-15-56/stdout.log",
    "Phase 2 (Energy)":    "/media/helios/Masters/Repository/dcase-few-shot-bioacoustic/baselines/dcase2024_task5/logs/experiments/runs/phase2_energy_seed1234/2026-09-18_16-56-51/stdout.log",
    "Phase 3 (Activation)": "/media/helios/Masters/Repository/dcase-few-shot-bioacoustic/baselines/dcase2024_task5/logs/experiments/runs/phase3_seed1234_output.log",
    "AFRI (Direct)": "/media/helios/Masters/Repository/dcase-few-shot-bioacoustic/baselines/dcase2024_task5/logs/experiments/runs/afri_direct_seed1234/2026-09-19_14-20-53/stdout.log",
    "AFRI (Inverse)": "/media/helios/Masters/Repository/dcase-few-shot-bioacoustic/baselines/dcase2024_task5/logs/experiments/runs/afri_inverse_seed1234/2026-09-19_20-45-13/stdout.log",
}

# Rounding used throughout the notebook (typical for a research paper).
N_DECIMALS = 2

# Filter out None entries for parsing
logs_to_parse = {k: v for k, v in EXPERIMENT_LOGS.items() if v is not None}
print(f"Parsing {len(logs_to_parse)} experiment(s)...")

# ── Paper tables output directory ──────────────────────────
# Every table produced below is also written as a stand-alone .tex file
# (\begin{table} block) under paper/tables/.
PAPER_TABLES_DIR = Path("../../../paper/tables").resolve()
PAPER_TABLES_DIR.mkdir(parents=True, exist_ok=True)

def save_tex(df, filename, caption, label, **to_latex_kwargs):
    """Write `df` as a standalone LaTeX table to PAPER_TABLES_DIR/filename."""
    kwargs = dict(
        float_format=f"%.{N_DECIMALS}f",
        na_rep="—",
        caption=caption,
        label=label,
    )
    kwargs.update(to_latex_kwargs)
    out = PAPER_TABLES_DIR / filename
    out.write_text(df.to_latex(**kwargs))
    print(f"  ✓ Saved {out.name}")


Parsing 8 experiment(s)...


## 1 — Load & Parse Logs

In [2]:
experiments = pl.load_experiments(logs_to_parse)

  ✓ Baseline                        fmeasure=16.615
  ✓ CIFE                            fmeasure=15.296
  ✓ Slot Uniform                    fmeasure=15.392
  ✓ Phase 1 (Random)                fmeasure=15.618
  ✓ Phase 2 (Energy)                fmeasure=15.434
  ✓ Phase 3 (Activation)            fmeasure=15.304
  ✓ AFRI (Direct)                   fmeasure=15.502
  ✓ AFRI (Inverse)                  fmeasure=15.613


## 2 — Summary Table (Main results)

All values are on a 0–100 scale.
- Precision / Recall come from the log on a 0–1 scale — rescaled here to 0–100 for consistency.
- F-measure is already reported on a 0–100 scale.
- Rounded to `N_DECIMALS` decimal places.

In [3]:
import pandas as pd
import numpy as np

df_final = pl.final_results_df(experiments)

# Preserve configured experiment order (missing runs become NaN rows)
exp_order = [k for k in EXPERIMENT_LOGS.keys() if k in df_final.index]
df_final = df_final.reindex(exp_order)

# Canonical column order (only keep what's available)
wanted = ["fmeasure", "fmeasureavg", "precision", "recall", "precisionavg", "recallavg"]
wanted = [c for c in wanted if c in df_final.columns]

RENAME = {
    "fmeasure":      "F-measure (Harmonic)",
    "fmeasureavg":  "F-measure (Arithmetic)",
    "precision":     "Precision (Harmonic)",
    "recall":        "Recall (Harmonic)",
    "precisionavg": "Precision (Arithmetic)",
    "recallavg":    "Recall (Arithmetic)",
}
df_summary = df_final[wanted].rename(columns=RENAME).copy()

# Rescale P / R columns from 0–1 to 0–100 to match the F-measure scale.
for col in ["Precision (Harmonic)", "Recall (Harmonic)", "Precision (Arithmetic)", "Recall (Arithmetic)"]:
    if col in df_summary.columns:
        col_max = df_summary[col].dropna().max()
        if pd.notna(col_max) and col_max <= 1.01:
            df_summary[col] = df_summary[col] * 100

df_summary = df_summary.round(N_DECIMALS)

print("=== Table 1 — Final Best Result Summary (0–100 scale) ===")
display(df_summary)

save_tex(
    df_summary,
    "results_summary.tex",
    caption=(
        "Comparison of experiment configurations on the DCASE 2024 Task~5 "
        "validation set. Both F-measure columns aggregate the six per-subset scores "
        "(each pooled from the TP/FP/FN of its subset's files); they differ only in "
        "the mean used -- harmonic (the official DCASE ranking metric) or arithmetic. "
        "All values on a 0--100 scale."
    ),
    label="tab:results_summary",
    bold_rows=True,
)

=== Table 1 — Final Best Result Summary (0–100 scale) ===


,F-measure (Harmonic),F-measure (Arithmetic),Precision (Harmonic),Recall (Harmonic),Precision (Arithmetic),Recall (Arithmetic)
experiment,,,,,,
Baseline,16.61,28.49,12.10,26.48,29.15,53.37
CIFE,15.30,29.52,12.39,19.98,28.47,53.49
Slot Uniform,15.39,27.54,10.61,27.99,25.42,54.13
Phase 1 (Random),15.62,27.65,10.80,28.20,25.80,54.31
Phase 2 (Energy),15.43,27.48,10.54,28.83,25.53,54.43
Phase 3 (Activation),15.30,27.74,11.48,22.95,26.33,52.47
AFRI (Direct),15.50,27.87,10.99,26.32,25.73,54.03
AFRI (Inverse),15.61,27.28,10.74,28.58,25.43,53.97


  ✓ Saved results_summary.tex


## 3 — F-measure Ranking (Harmonic Subset Mean)

F-measure (Harmonic) is the **harmonic mean** of the six per-subset F-measures, each computed from counts pooled within its own subset. This is the metric DCASE ranks on.

In [4]:
if "F-measure (Harmonic)" in df_summary.columns:
    df_harm = (
        df_summary[["F-measure (Harmonic)"]]
        .dropna()
        .sort_values("F-measure (Harmonic)", ascending=False)
    )
    df_harm.insert(0, "Rank", np.arange(1, len(df_harm) + 1))

    # Δ vs the top-ranked experiment
    top = df_harm["F-measure (Harmonic)"].iloc[0]
    df_harm["Δ vs. best"] = (df_harm["F-measure (Harmonic)"] - top).round(N_DECIMALS)

    print("=== Table 2 — F-measure Ranking (Harmonic Subset Mean) ===")
    display(df_harm)

    save_tex(
        df_harm,
        "results_ranking_harmonic.tex",
        caption=(
            "Ranking of experiment configurations by F-measure, aggregated as the "
            "harmonic mean of the six per-subset scores. "
            "$\\Delta$ vs. best is the absolute difference to the top-ranked "
            "configuration. All values on a 0--100 scale."
        ),
        label="tab:results_ranking_harmonic",
        bold_rows=True,
    )
else:
    print("Harmonic subset-mean F-measure not available in parsed results.")

=== Table 2 — F-measure Ranking (Harmonic Subset Mean) ===


,Rank,F-measure (Harmonic),Δ vs. best
experiment,,,
Baseline,1,16.61,0.00
Phase 1 (Random),2,15.62,-0.99
AFRI (Inverse),3,15.61,-1.00
AFRI (Direct),4,15.50,-1.11
Phase 2 (Energy),5,15.43,-1.18
Slot Uniform,6,15.39,-1.22
CIFE,7,15.30,-1.31
Phase 3 (Activation),8,15.30,-1.31


  ✓ Saved results_ranking_harmonic.tex


## 4 — F-measure Ranking (Arithmetic Subset Mean)

F-measure (Arithmetic) is the **unweighted arithmetic mean** of those same six per-subset F-measures.

In [5]:
if "F-measure (Arithmetic)" in df_summary.columns:
    df_arith = (
        df_summary[["F-measure (Arithmetic)"]]
        .dropna()
        .sort_values("F-measure (Arithmetic)", ascending=False)
    )
    df_arith.insert(0, "Rank", np.arange(1, len(df_arith) + 1))

    top = df_arith["F-measure (Arithmetic)"].iloc[0]
    df_arith["Δ vs. best"] = (df_arith["F-measure (Arithmetic)"] - top).round(N_DECIMALS)

    print("=== Table 3 — F-measure Ranking (Arithmetic Subset Mean) ===")
    display(df_arith)

    save_tex(
        df_arith,
        "results_ranking_arithmetic.tex",
        caption=(
            "Ranking of experiment configurations by F-measure, aggregated as the "
            "unweighted arithmetic mean of the six per-subset scores. "
            "$\\Delta$ vs. best is the absolute difference to the top-ranked "
            "configuration. All values on a 0--100 scale."
        ),
        label="tab:results_ranking_arithmetic",
        bold_rows=True,
    )
else:
    print("Arithmetic subset-mean F-measure not available in parsed results.")

=== Table 3 — F-measure Ranking (Arithmetic Subset Mean) ===


,Rank,F-measure (Arithmetic),Δ vs. best
experiment,,,
CIFE,1,29.52,0.00
Baseline,2,28.49,-1.03
AFRI (Direct),3,27.87,-1.65
Phase 3 (Activation),4,27.74,-1.78
Phase 1 (Random),5,27.65,-1.87
Slot Uniform,6,27.54,-1.98
Phase 2 (Energy),7,27.48,-2.04
AFRI (Inverse),8,27.28,-2.24


  ✓ Saved results_ranking_arithmetic.tex


## 5 — Precision / Recall / F-measure per Experiment (Harmonic Subset Mean)

Tabular equivalent of the grouped bar chart in `results_comparison.ipynb`.

In [6]:
prf_cols = [c for c in ["Precision (Harmonic)", "Recall (Harmonic)", "F-measure (Harmonic)"]
            if c in df_summary.columns]

if prf_cols:
    df_prf = df_summary[prf_cols].copy()
    print("=== Table 4 — Precision / Recall / F-measure (Harmonic Subset Mean, 0–100 scale) ===")
    display(df_prf)

    save_tex(
        df_prf,
        "results_prf_harmonic.tex",
        caption=(
            "Precision, recall, and F-measure per experiment, each aggregated as the "
            "harmonic mean over the six per-subset scores, "
            "on the DCASE 2024 Task~5 validation set. All values on a 0--100 scale."
        ),
        label="tab:results_prf_harmonic",
        bold_rows=True,
    )
else:
    print("Precision/Recall/F-measure columns not available.")

=== Table 4 — Precision / Recall / F-measure (Harmonic Subset Mean, 0–100 scale) ===


,Precision (Harmonic),Recall (Harmonic),F-measure (Harmonic)
experiment,,,
Baseline,12.10,26.48,16.61
CIFE,12.39,19.98,15.30
Slot Uniform,10.61,27.99,15.39
Phase 1 (Random),10.80,28.20,15.62
Phase 2 (Energy),10.54,28.83,15.43
Phase 3 (Activation),11.48,22.95,15.30
AFRI (Direct),10.99,26.32,15.50
AFRI (Inverse),10.74,28.58,15.61


  ✓ Saved results_prf_harmonic.tex


## 6 — Effect of Post-processing

- **No Post-proc F**: best F-measure (Harmonic) across detection thresholds *before* any post-processing.
- **Best Post-proc F**: final F-measure (Harmonic) after post-processing threshold selection (same value as in Table 1).
- **Gain**: absolute improvement contributed by post-processing.

In [7]:
df_nopost = pl.no_postproc_df(experiments)

if not df_nopost.empty and "F-measure (Harmonic)" in df_summary.columns:
    no_post_best = (
        df_nopost.groupby("experiment")["fmeasure"]
        .max()
        .reindex(exp_order)
    )
    post_best = df_summary["F-measure (Harmonic)"].reindex(exp_order)

    df_postproc = pd.DataFrame({
        "No Post-proc F":   no_post_best.values,
        "Best Post-proc F": post_best.values,
        "Gain":             post_best.values - no_post_best.values,
    }, index=exp_order).round(N_DECIMALS)

    print("=== Table 5 — Post-processing Effect on F-measure (Harmonic Subset Mean) ===")
    display(df_postproc)

    save_tex(
        df_postproc,
        "results_postproc_effect.tex",
        caption=(
            "Effect of post-processing on the F-measure (harmonic mean over the six "
            "per-subset scores). "
            "\\emph{No Post-proc F} is the best F over detection thresholds "
            "$k$ before post-processing; \\emph{Best Post-proc F} is the final "
            "F after post-processing threshold selection; \\emph{Gain} is the "
            "absolute improvement contributed by post-processing. "
            "All values on a 0--100 scale."
        ),
        label="tab:results_postproc_effect",
        bold_rows=True,
    )
else:
    print("No-postproc data not available (check log parsing).")

=== Table 5 — Post-processing Effect on F-measure (Harmonic Subset Mean) ===


,No Post-proc F,Best Post-proc F,Gain
Baseline,11.29,16.61,5.32
CIFE,9.09,15.30,6.21
Slot Uniform,10.44,15.39,4.95
Phase 1 (Random),10.26,15.62,5.36
Phase 2 (Energy),10.70,15.43,4.73
Phase 3 (Activation),10.27,15.30,5.03
AFRI (Direct),10.38,15.50,5.12
AFRI (Inverse),10.92,15.61,4.69


  ✓ Saved results_postproc_effect.tex


## 7 — Harmonic vs. Arithmetic Subset-Mean F-measure

Tabular equivalent of the scatter plot. **Gap** = Harmonic − Arithmetic, which is always $\le 0$ by the AM–HM inequality. Its *magnitude* measures how unevenly performance is spread across the six subsets: a large negative gap means one subset is far weaker than the rest. It says nothing about file sizes or class balance.

In [8]:
if {"F-measure (Harmonic)", "F-measure (Arithmetic)"}.issubset(df_summary.columns):
    df_mm = df_summary[["F-measure (Harmonic)", "F-measure (Arithmetic)"]].copy()
    df_mm["Gap (Harm. − Arith.)"] = (
        df_mm["F-measure (Harmonic)"] - df_mm["F-measure (Arithmetic)"]
    ).round(N_DECIMALS)

    print("=== Table 6 — Harmonic vs. Arithmetic Subset-Mean F-measure ===")
    display(df_mm)

    save_tex(
        df_mm,
        "results_harmonic_vs_arithmetic.tex",
        caption=(
            "The two subset-mean F-measures with the signed gap "
            "(\\emph{Harmonic} $-$ \\emph{Arithmetic}), which is always non-positive "
            "by the AM--HM inequality. The magnitude of the gap measures how unevenly "
            "the six per-subset scores are spread: a large gap means one subset is far "
            "weaker than the rest. "
            "All values on a 0--100 scale."
        ),
        label="tab:results_harmonic_vs_arithmetic",
        bold_rows=True,
    )
else:
    print("Both the harmonic and arithmetic subset-mean F-measures are required for this table.")

=== Table 6 — Harmonic vs. Arithmetic Subset-Mean F-measure ===


,F-measure (Harmonic),F-measure (Arithmetic),Gap (Harm. − Arith.)
experiment,,,
Baseline,16.61,28.49,-11.88
CIFE,15.30,29.52,-14.22
Slot Uniform,15.39,27.54,-12.15
Phase 1 (Random),15.62,27.65,-12.03
Phase 2 (Energy),15.43,27.48,-12.05
Phase 3 (Activation),15.30,27.74,-12.44
AFRI (Direct),15.50,27.87,-12.37
AFRI (Inverse),15.61,27.28,-11.67


  ✓ Saved results_harmonic_vs_arithmetic.tex


## 8 — LaTeX Export (Main summary table)

Copy-paste ready `\begin{table}` block for a research paper. All values are on a 0–100 scale, rounded to two decimals.

In [9]:
latex_str = df_summary.to_latex(
    float_format=f"%.{N_DECIMALS}f",
    na_rep="—",
    caption=(
        "Comparison of experiment configurations on DCASE 2024 Task 5 validation set. "
        "Both F-measure columns aggregate the six per-subset scores; the first uses a "
        "harmonic mean (the official DCASE metric), the second an arithmetic mean. "
        "All values on a 0--100 scale."
    ),
    label="tab:results_comparison",
    bold_rows=True,
)
print(latex_str)

\begin{table}
\centering
\caption{Comparison of experiment configurations on DCASE 2024 Task 5 validation set. Both F-measure columns aggregate the six per-subset scores; the first uses a harmonic mean (the official DCASE metric), the second an arithmetic mean. All values on a 0--100 scale.}
\label{tab:results_comparison}
\begin{tabular}{lrrrrrr}
\toprule
{} &  F-measure (Harmonic) &  F-measure (Arithmetic) &  Precision (Harmonic) &  Recall (Harmonic) &  Precision (Arithmetic) &  Recall (Arithmetic) \\
\textbf{experiment          } &                       &                         &                       &                    &                         &                      \\
\midrule
\textbf{Baseline            } &                 16.61 &                   28.49 &                 12.10 &              26.48 &                   29.15 &                53.37 \\
\textbf{CIFE                } &                 15.30 &                   29.52 &                 12.39 &              19.98 &   